## **Proyecto: Detección de Cambios en Interfaces Web para RPA usando IA**

## **2. Pipeline de Limpieza de Datos**

Para nuestro proyecto, vamos a entrenar una Red Neuronal YOLOV8.

Nuestro dataset, es diferente a un dataset tabular clásico (Con columnas numéricas, categóricas, filas, etc..) en YOLO el datset tiene un formato específico y consiste en:
- Imágenes (.jpg, .png …).
- Labels (.txt) con anotaciones en formato YOLO: class x_center y_center width height.
- dataset.yaml con paths y clases.

Por esta razón en análisis y limpieza de datos debe abordarse de manera que se adapte al tipo de datos (imágenes y anotaciones)

### Estructura de la raiz del Dataset
```
Dataset/
├── train/
│    ├── images/ imagenes.png ...
│    ├── labels/ architos.txt ...
├── val/
│    ├── images/ imagenes.png ...
│    ├── labels/ architos.txt ...
└── test/
│    ├── images/ imagenes.png ...
│    ├── labels/ architos.txt ...
└── dataset.yaml (archivo de configuración)

In [1]:
# 1. Montar Google Drive para usar el dataset desde ahi
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
repo_path = "/content/drive/MyDrive/MIA/Dataset" #Ruta de la carpeta Raiz del dataset

### **2.1 Tratamiento de Valores Faltantes**
**Estrategias:**
- Eliminar imágenes sin anotaciones (.txt) y anotaciones sin imágenes.
- Detectar labels vacíos o con formato incorrecto.
- Imputación no aplica directamente, pero se puede:
    - Usar "indicadores de faltantes" → por ejemplo, marcar qué imágenes no tienen anotaciones.
    - Opcional: mover esas imágenes, sacarlas del dataset a una carpeta "faltantes" si es necesario.

### **2.2 Tratamiento de Outliers**
Para un conjunto de datos en el contexto de **YOLOv8**, un outlier o valor atípico, podría ser es una anotación de caja delimitadora (bounding box) o una instancia de un objeto que se desvía significativamente de las características comunes del resto de los datos. Esto puede ser en términos de tamaño, forma, posición, o incluso la clase a la que se le asigna. En esencia, es un dato inusual que no sigue el patrón general.

**Detección automatizada:**
- Bounding boxes con coordenadas fuera de [0,1].
- Anotaciones con width o height muy pequeños o demasiado grandes (p. ej. <0.01 o >0.9).

**Estrategias para los Outliers:**
- Eliminación de labels inválidos.
- Winsorizing/capping → recortar valores extremos para que estén dentro de [0,1].
- Opcional: mover imágenes con anomalías a un subconjunto aparte.

### **2.3 Estandarizar Formatos**
Estandarizar un dataset para **YOLOv8** implica asegurar que todas las imágenes y sus correspondientes anotaciones sigan un formato y una estructura consistentes. Esto es crucial para que el modelo de aprendizaje automático pueda procesar los datos de manera uniforme y efectiva.

**Tipos de datos correctos:**
- Contenido del archivo .txt
  - Cada línea del archivo .txt representa un objeto detectado y contiene 5 valores: clase_id, x_centro, y_centro, ancho, alto.
- Clases enteras (int) (Son las categorías que se van a detectar; por ejemplo: 0 link, 1 botón, etc...)

**Consistencia en categorías:**
- Todas las clases deben estar en dataset.yaml.

**Normalización:**
- Revisar que todas las imágenes tengan formato .jpg o .png.
- Los valores de las coordenadas (x_centro, y_centro, ancho, alto) están normalizados entre 0 y 1. Esto significa que se dividen por el ancho y el alto de la imagen, respectivamente (El dataset utilizado ya tiene estas cajas delimitadoras -bounding boxes- definidas, pero hay que considerar esta normalización si se agregan nuevos datos al dataset)
  - Xcentro = Xpixel / Ancho Imagen
  - ycentro = Ypixel / Alto Imagen
  - Ancho caja delimitadora = Ancho Pixel / Ancho Imagen
  - Alto caja delimitador = Alto Pixel / Alto Imagen


### **2.4 Pipeline Automatizado**

In [34]:
import os
import glob
import numpy as np
import shutil

class DataCleaner:
    def __init__(self, repo_path, nclasesmax):
        self.repo_path = repo_path
        #cantidad de clases definidas en .yaml ej: 0: link, 1:buton, .. 15: text; entonces nclasesmax = 15
        self.nclasesmax = nclasesmax
        self.missing = []
        self.outliers = []
    #2.1 Tratamiento de Valores Faltantes
    def _check_missing(self, img_dir, label_dir, moverfaltantes = False):
      #Detecta imágenes sin label y labels sin imagen
      #para cada directorio extrae el nombre base del archivo en minúscula
      images = {os.path.splitext(f)[0] for f in os.listdir(img_dir) if f.lower().endswith((".jpg", ".png"))}
      labels = {os.path.splitext(f)[0] for f in os.listdir(label_dir) if f.lower().endswith(".txt")}
      #se utilizaran los mismos nombres de archivos, cada imagen .png o .jpg debe tener su archivo .txt con el mismo nombre
      #se realiza una operación de conjuntos
      missing_labels = images - labels #imagenes que no tiene su archivo txt con etiquetas
      missing_images = labels - images #archivos txt que no tienen su imagen
      if(moverfaltantes): #opcional mover faltantes
        carpetapadr = os.path.basename(os.path.dirname(img_dir))
        pathfaltante = os.path.join(repo_path,carpetapadr,"faltantes")
        if not os.path.exists(pathfaltante):
          os.makedirs(pathfaltante) #si no existe la carpeta faltantes la creo
        missing_img_dir = os.path.join(pathfaltante, "falta_labels")
        missing_lbl_dir = os.path.join(pathfaltante, "falta_imagen")
        if not os.path.exists(missing_img_dir):
          os.makedirs(missing_img_dir)
        if not os.path.exists(missing_lbl_dir):
          os.makedirs(missing_lbl_dir)

      if missing_labels:
          #añado la tupla nombre_imagen, "falta el archivo txt"
          self.missing.extend([(img, "falta el archito txt") for img in missing_labels])
          if(moverfaltantes):
            print(f"Moviendo {len(missing_labels)} imágenes sin etiquetas...")
            for img_base in missing_labels:
              source_img = os.path.join(img_dir, f"{img_base}.png") # Asumimos .png, se usará png para facilitar el proceso
              dest_img = os.path.join(missing_img_dir, f"{img_base}.jpg")
              shutil.move(source_img, dest_img)
      if missing_images:
          #añado la tupla archivo_txt, "falta imagen"
          self.missing.extend([(lbl, "missing_image") for lbl in missing_images])
          if(moverfaltantes):
            print(f"Moviendo {len(missing_images)} archivos txt sin imágenes...")
            for lbl_base in missing_images:
              source_lbl = os.path.join(label_dir, f"{lbl_base}.txt")
              dest_lbl = os.path.join(missing_lbl_dir, f"{lbl_base}.txt")
              shutil.move(source_lbl, dest_lbl)

    #2.2 Tratamiento de Outliers y 2.3 estandarizar formatos Revisión
    def _check_outliers(self, label_dir):
      #Detecta anotaciones fuera de rango o mal formateadas
      #proceso cada archivo txt (estos contienen las etiquetas en formato YOLO para cada imagen)
      for txt in glob.glob(os.path.join(label_dir, "*.txt")):
        with open(txt, "r") as f: #Abro el archivo para lectura
          for i,line in enumerate(f,1): #itero en cada línea dentro del archivo
            try:
              parts = line.strip().split() #divido la línea en partes
              if len(parts) != 5: #un fomrato YOLOV8 válido debe tener exactamente 5 partes (clase, x, y, w, h)
                self.outliers.append((txt, "formato inválido (linea no cumple 5 partes)", f"Línea {i}: {line.strip()}"))
                continue #salto a la siguiente linea

              cls, x, y, w, h = map(float, parts) #si esta conversión falla quiere decir que no son valores nunéricos float

              if not (0 <= cls <= self.nclasesmax):# Validar que sea una clase válida
                self.outliers.append((txt, f"Clase inválida (debe ser entre 0 y {self.nclasesmax})", f"Línea {i}: {line.strip()}"))
                continue

              if not (0 <= x <= 1 and 0 <= y <= 1 and 0 < w <= 1 and 0 < h <= 1):
                self.outliers.append((txt, "fuera de rango (revisar normalizacion)", f"Línea {i}: {line.strip()}"))
            except:
              self.outliers.append((txt, "error (no se puede mapear)", f"Línea {i}: {line.strip()}"))

    def fit(self, moverarchivosincompletos = False):
        #Escanea la estructura de un dataset YOLOv8 train/val/test en busca de problemas
        for subset in ["train", "val", "test"]:
            img_dir = os.path.join(self.repo_path, subset, "images")
            label_dir = os.path.join(self.repo_path, subset, "labels")

            if os.path.exists(img_dir) and os.path.exists(label_dir):
                self._check_missing(img_dir, label_dir, moverarchivosincompletos)
                self._check_outliers(label_dir)
        return self

    # Aplicar Opcional Limpieza
    def transform(self, removerLabelsinvalidos=False):
        #Acción sobre los problemas encontrados.
        if removerLabelsinvalidos: #borra el archivo de labels(habría que volverlo a generar pero bien)
            for file, issue, line in self.outliers:
                print(f"remover label inválido: {file} | {issue} | {line}")
                os.remove(file)
        return self

    def fit_transform(self, removerLabelsinvalidos=False, moverarchivosincompletos=False):
        return self.fit(moverarchivosincompletos).transform(removerLabelsinvalidos)



In [36]:
repo_path = "/content/drive/MyDrive/MIA/Dataset"
limpieza = DataCleaner(repo_path,nclasesmax= 15)
limpieza.fit_transform(removerLabelsinvalidos=False,moverarchivosincompletos=False)
print("**Resumen de limpieza**")
print("- Faltantes:", limpieza.missing if limpieza.missing else "Ninguno")
print("- Outliers:", limpieza.outliers if limpieza.outliers else "Ninguno")


**Resumen de limpieza**
- Faltantes: Ninguno
- Outliers: Ninguno
